In [10]:
import pandas as pd

def load_and_clean_data(file_path):
    df = pd.read_csv(file_path)

    df["timestamp"] = pd.to_datetime(df["timestamp"])

    df = df.sort_values(
        by=["patient_id", "timestamp"]
    )

    df = df.reset_index(drop=True)

    return df


In [11]:
df = load_and_clean_data(r"C:\Academics\vs code\Projects\Clinical-Trial-Mining\data\dataset.csv")

In [ ]:
df.describe()

In [13]:
df.head(5)

,patient_id,site_id,timestamp,event,deviation_flag,deviation_severity
0,P00001,S11,2025-09-05,Consent Signed,0,NaN
1,P00001,S11,2025-09-11,Visit Scheduled,0,NaN
2,P00001,S11,2025-09-12,Visit Completed,0,NaN
3,P00001,S11,2025-09-19,Sample Collected,0,NaN
4,P00001,S11,2025-09-28,Lab Ordered,0,NaN


In [ ]:
df['deviation_severity']

In [14]:
def build_sequences(df):

    sequences = (
        df.groupby("patient_id")["event"]
        .apply(list)
        .tolist()
    )

    return sequences


In [15]:
from prefixspan import PrefixSpan

def mine_patterns(sequences, min_support=50):
    ps = PrefixSpan(sequences)
    patterns = ps.frequent(min_support)
    return patterns


In [ ]:
print(df["event"].value_counts())

In [16]:
sequences = build_sequences(df)

In [17]:
patterns = mine_patterns(sequences, 100)

In [ ]:
print(patterns[:20])

In [ ]:
for support, pattern in patterns[:20]:
    print(f"Support: {support}, Length: {len(pattern)}, Pattern: {pattern}")

In [ ]:
lengths = [len(pattern) for support, pattern in patterns]

print("Min Length:", min(lengths))
print("Max Length:", max(lengths))
print("Average Length:", sum(lengths)/len(lengths))

In [18]:
filtered_patterns = [(support, pattern) for support, pattern in patterns if len(pattern) >= 2]

In [ ]:
len(filtered_patterns)

In [ ]:
lengths = [len(patterns) for support, pattern in patterns]

print("Min Length:", min(lengths))
print("Max Length:", max(lengths))
print("Average Length:", sum(lengths)/len(lengths))

In [ ]:
print(filtered_patterns[:5])

In [ ]:
for support, pattern in filtered_patterns[:20]:
    print(
        f"Support={support}, "
        f"Length={len(pattern)}, "
        f"Pattern={pattern}"
    )

In [19]:
deviation_patterns = [
    (support, pattern)
    for support, pattern in filtered_patterns
    if pattern[-1] == "Protocol Deviation"
]

In [20]:
deviation_patterns.sort(
    key=lambda x: x[0],
    reverse=True
)

In [21]:
for support, pattern in deviation_patterns[:20]:
    print(
        f"Support={support}, "
        f"Length={len(pattern)}, "
        f"Pattern={pattern}"
    )

Support=3319, Length=2, Pattern=['Query Raised', 'Protocol Deviation']
Support=1738, Length=2, Pattern=['Consent Signed', 'Protocol Deviation']
Support=1690, Length=3, Pattern=['Consent Signed', 'Query Raised', 'Protocol Deviation']
Support=1688, Length=3, Pattern=['Consent Signed', 'Visit Scheduled', 'Protocol Deviation']
Support=1688, Length=2, Pattern=['Visit Scheduled', 'Protocol Deviation']
Support=1640, Length=4, Pattern=['Consent Signed', 'Visit Scheduled', 'Query Raised', 'Protocol Deviation']
Support=1640, Length=3, Pattern=['Visit Scheduled', 'Query Raised', 'Protocol Deviation']
Support=1420, Length=2, Pattern=['Missed Visit', 'Protocol Deviation']
Support=1390, Length=3, Pattern=['Missed Visit', 'Query Raised', 'Protocol Deviation']
Support=936, Length=3, Pattern=['Consent Signed', 'Missed Visit', 'Protocol Deviation']
Support=925, Length=3, Pattern=['Consent Signed', 'Visit Delayed', 'Protocol Deviation']
Support=925, Length=2, Pattern=['Visit Delayed', 'Protocol Deviation

In [24]:
total_sequences = len(sequences)

patterns_with_pct = pd.DataFrame([
    (
        support,
        round(support * 100 / total_sequences, 2),
        pattern
    )
    for support, pattern in deviation_patterns
])

In [ ]:
for support, pct, pattern in patterns_with_pct[:10]:
    print(
        f"{pct}% | {support} | {pattern}"
    )

In [25]:
patterns_with_pct.head(5)

,0,1,2
0,3319,41.49,"[Query Raised, Protocol Deviation]"
1,1738,21.73,"[Consent Signed, Protocol Deviation]"
2,1690,21.12,"[Consent Signed, Query Raised, Protocol Deviat..."
3,1688,21.10,"[Consent Signed, Visit Scheduled, Protocol Dev..."
4,1688,21.10,"[Visit Scheduled, Protocol Deviation]"
